In [0]:
%sql
CREATE OR REPLACE TABLE hub_sbx.s_ipd.s_bowler_billing_master_ww_ipd
USING DELTA
AS

/* ===== CTEs for pre-filtering & type normalization ===== */

WITH us_ipd AS (

    SELECT
        CAST(SUBSTRING_INDEX(u.MaterialKey, '|', -1) AS STRING) AS MaterialId,
        CAST(u.MaterialKey AS STRING)                       AS MaterialKey,
        CAST(u.ErpCustomerKey AS STRING)                    AS ErpCustomerKey,
        CAST(u.DistributorKey AS STRING)                    AS DistributorKey,
        CAST(u.ShipToAccountNumber AS STRING)               AS ShipToAccountNumber,
        CAST(u.SoldToAccountNumber AS STRING)               AS SoldToAccountNumber,
        CAST(u.ReportingDate AS DATE)                         AS InvoiceDate,
        CAST(u.IndirectSellDate AS DATE)                    AS IndirectSellDate,
        CAST(u.DateAdded AS DATE)                           AS DateAdded,
        CAST(u.BillingTypeDescription AS STRING)            AS BillingTypeDescription,
        CAST(u.BillingQuantity AS DECIMAL(38,10))           AS BillingQuantity,
        CAST(u.InvoiceQuantity AS DECIMAL(38,10))           AS InvoiceQuantity,
        CAST(u.ActualNetRevenue AS DECIMAL(38,10))          AS ActualNetRevenue,
        CAST(u.DistributionChannel AS STRING)               AS DistributionChannel,
        CAST(u.TransactionType AS STRING)                   AS TransactionType,
        CAST(u.ActualGrossProfit AS DECIMAL(38,10))         AS ActualGrossProfit,
        CAST(u.Country AS STRING)                           AS Country,
        CAST(u.LocalCurrency AS STRING)                     AS Currency,
        CAST(u.DirectSalesViewFlag AS BOOLEAN)              AS DirectSalesViewFlag,
        CAST(u.CustomerViewFlag AS BOOLEAN)                 AS CustomerViewFlag,
        CAST(u.SalesFlag AS BOOLEAN)                        AS SalesFlag
    FROM hub_prd.s_commops_na.billing_master_us u
    INNER JOIN hub_prd.s_commops_na.unified_material_master_na m
        ON u.MaterialKey = m.MaterialKey
    WHERE m.ReltioPHLevel1 IN (
        'Alaris System',
        'Hazardous Drug Solutions',
        'IV Solutions',
        'Gravity and Syringe Delivery',
        'IV Access',
        'CME',
        'Alaris Plus',
        'IPD OEM',
        'BD Nexus'
    )

),

ca_ipd AS (

    SELECT
        CAST(SUBSTRING_INDEX(c.MaterialKey, '|', -1) AS STRING) AS MaterialId,
        CAST(c.MaterialKey AS STRING)                       AS MaterialKey,
        CAST(c.ErpCustomerKey AS STRING)                    AS ErpCustomerKey,
        CAST(c.DistributorKey AS STRING)                    AS DistributorKey,
        CAST(c.ShipToAccountNumber AS STRING)               AS ShipToAccountNumber,
        CAST(c.SoldToAccountNumber AS STRING)               AS SoldToAccountNumber,
        CAST(c.InvoiceDate AS DATE)                         AS InvoiceDate,
        CAST(c.IndirectSellDate AS DATE)                    AS IndirectSellDate,
        CAST(c.DateAdded AS DATE)                           AS DateAdded,
        CAST(c.BillingTypeDescription AS STRING)            AS BillingTypeDescription,
        CAST(c.BillingQuantity AS DECIMAL(38,10))           AS BillingQuantity,
        CAST(c.InvoiceQuantity AS DECIMAL(38,10))           AS InvoiceQuantity,
        CAST(c.ActualNetRevenue AS DECIMAL(38,10))          AS ActualNetRevenue,
        CAST(c.DistributionChannel AS STRING)               AS DistributionChannel,
        CAST(c.TransactionType AS STRING)                   AS TransactionType,
        CAST(c.ActualGrossProfit AS DECIMAL(38,10))         AS ActualGrossProfit,
        CAST(c.Country AS STRING)                           AS Country,
        CAST(c.LocalCurrency AS STRING)                     AS Currency,
        CAST(c.DirectSalesViewFlag AS BOOLEAN)              AS DirectSalesViewFlag,
        CAST(c.CustomerViewFlag AS BOOLEAN)                 AS CustomerViewFlag,
        CAST(c.SalesFlag AS BOOLEAN)                        AS SalesFlag
    FROM hub_sbx.s_ipd.s_billing_master_ca_ipd c

),

ga_ipd AS (

    SELECT
        CAST(g.MaterialId AS STRING)                        AS MaterialId,
        CAST(g.MaterialKey AS STRING)                       AS MaterialKey,
        CAST(g.ErpCustomerKey AS STRING)                    AS ErpCustomerKey,
        CAST(g.DistributorKey AS STRING)                    AS DistributorKey,
        CAST(g.ShipToAccountNumber AS STRING)               AS ShipToAccountNumber,
        CAST(g.SoldToAccountNumber AS STRING)               AS SoldToAccountNumber,
        CAST(g.InvoiceDate AS DATE)                         AS InvoiceDate,
        CAST(g.IndirectSellDate AS DATE)                    AS IndirectSellDate,
        CAST(g.DateAdded AS DATE)                           AS DateAdded,
        CAST(g.BillingTypeDescription AS STRING)            AS BillingTypeDescription,
        CAST(g.BillingQuantity AS DECIMAL(38,10))           AS BillingQuantity,
        CAST(g.InvoiceQuantity AS DECIMAL(38,10))           AS InvoiceQuantity,
        CAST(g.ActualNetRevenue AS DECIMAL(38,10))          AS ActualNetRevenue,
        CAST(g.DistributionChannel AS STRING)               AS DistributionChannel,
        CAST(g.TransactionType AS STRING)                   AS TransactionType,
        CAST(g.ActualGrossProfit AS DECIMAL(38,10))         AS ActualGrossProfit,
        CAST(g.Country AS STRING)                           AS Country,
        CAST(g.LocalCurrency AS STRING)                     AS Currency,
        CAST(g.DirectSalesViewFlag AS BOOLEAN)              AS DirectSalesViewFlag,
        CAST(g.CustomerViewFlag AS BOOLEAN)                 AS CustomerViewFlag,
        CAST(g.SalesFlag AS BOOLEAN)                        AS SalesFlag
    FROM hub_sbx.s_ipd.s_billing_master_ga_ipd g

),

latam_ipd AS (

    SELECT
        CAST(l.MaterialId AS STRING)                        AS MaterialId,
        CAST(m.MaterialKey AS STRING)                       AS MaterialKey,
        CAST(NULL AS STRING)                                AS ErpCustomerKey,
        CAST(NULL AS STRING)                                AS DistributorKey,
        CAST(l.ShipToAccountNumber AS STRING)               AS ShipToAccountNumber,
        CAST(l.SoldToAccountNumber AS STRING)               AS SoldToAccountNumber,
        CAST(l.InvoiceDate AS DATE)                         AS InvoiceDate,
        CAST(NULL AS DATE)                                  AS IndirectSellDate,
        CAST(l.DateAdded AS DATE)                           AS DateAdded,
        CAST(l.BillingType AS STRING)                       AS BillingTypeDescription,
        CAST(l.BillingQuantity AS DECIMAL(38,10))           AS BillingQuantity,
        CAST(NULL AS DECIMAL(38,10))                        AS InvoiceQuantity,
        CAST(l.ActualNetRevenue AS DECIMAL(38,10))          AS ActualNetRevenue,
        CAST(l.DistributionChannel AS STRING)               AS DistributionChannel,
        CAST(l.BillingType AS STRING)                       AS TransactionType,
        CAST(NULL AS DECIMAL(38,10))                        AS ActualGrossProfit,
        CAST(l.Country AS STRING)                           AS Country,
        CAST(l.LocalCurrency AS STRING)                     AS Currency,
        CAST(NULL AS BOOLEAN)                               AS DirectSalesViewFlag,
        CAST(NULL AS BOOLEAN)                               AS CustomerViewFlag,
        CAST(NULL AS BOOLEAN)                               AS SalesFlag
    FROM hub_prd.s_region_latam.billing_master_latam l
    INNER JOIN hub_prd.s_region_latam.unified_material_master_latam m
        ON  l.SourceSystemId = m.SourceSystemId
        AND l.MaterialId     = m.MaterialId
    WHERE m.ErpPhLevel1 IN (
        'Alaris System',
        'Hazardous Drug Solutions',
        'IV Solutions',
        'Gravity and Syringe Delivery',
        'IV Access',
        'CME',
        'Alaris Plus',
        'IPD OEM',
        'BD Nexus'
    )

)

/* ===== Final UNION ALL (aligned schema & types) ===== */

SELECT
    MaterialId,
    MaterialKey,
    ErpCustomerKey,
    DistributorKey,
    ShipToAccountNumber,
    SoldToAccountNumber,
    InvoiceDate,
    IndirectSellDate,
    DateAdded,
    BillingTypeDescription,
    BillingQuantity,
    InvoiceQuantity,
    ActualNetRevenue,
    DistributionChannel,
    TransactionType,
    ActualGrossProfit,
    Country,
    Currency,
    DirectSalesViewFlag,
    CustomerViewFlag,
    SalesFlag,
    'hub_prd.s_commops_na.billing_master_us_ipd' AS Source
FROM us_ipd

UNION ALL

SELECT
    MaterialId,
    MaterialKey,
    ErpCustomerKey,
    DistributorKey,
    ShipToAccountNumber,
    SoldToAccountNumber,
    InvoiceDate,
    IndirectSellDate,
    DateAdded,
    BillingTypeDescription,
    BillingQuantity,
    InvoiceQuantity,
    ActualNetRevenue,
    DistributionChannel,
    TransactionType,
    ActualGrossProfit,
    Country,
    Currency,
    DirectSalesViewFlag,
    CustomerViewFlag,
    SalesFlag,
    'hub_sbx.s_ipd.s_billing_master_ca_ipd' AS Source
FROM ca_ipd

UNION ALL

SELECT
    MaterialId,
    MaterialKey,
    ErpCustomerKey,
    DistributorKey,
    ShipToAccountNumber,
    SoldToAccountNumber,
    InvoiceDate,
    IndirectSellDate,
    DateAdded,
    BillingTypeDescription,
    BillingQuantity,
    InvoiceQuantity,
    ActualNetRevenue,
    DistributionChannel,
    TransactionType,
    ActualGrossProfit,
    Country,
    Currency,
    DirectSalesViewFlag,
    CustomerViewFlag,
    SalesFlag,
    'hub_sbx.s_ipd.s_billing_master_ga_ipd' AS Source
FROM ga_ipd

UNION ALL

SELECT
    MaterialId,
    MaterialKey,
    ErpCustomerKey,
    DistributorKey,
    ShipToAccountNumber,
    SoldToAccountNumber,
    InvoiceDate,
    IndirectSellDate,
    DateAdded,
    BillingTypeDescription,
    BillingQuantity,
    InvoiceQuantity,
    ActualNetRevenue,
    DistributionChannel,
    TransactionType,
    ActualGrossProfit,
    Country,
    Currency,
    DirectSalesViewFlag,
    CustomerViewFlag,
    SalesFlag,
    'hub_prd.s_region_latam.billing_master_latam_ipd' AS Source
FROM latam_ipd

%sql
CREATE OR REPLACE VIEW hub_sbx.g_ipd.g_bowler_billing_master_ww_ipd AS
SELECT
    MaterialId AS `Material Id`,
    MaterialKey AS `Material Key`,
    ErpCustomerKey AS `Erp Customer Key`,
    DistributorKey AS `Distributor Key`,
    ShipToAccountNumber AS `Ship To Account Number`,
    SoldToAccountNumber AS `Sold To Account Number`,
    InvoiceDate AS `Invoice Date`,
    IndirectSellDate AS `Indirect Sell Date`,
    DateAdded AS `Date Added`,
    BillingTypeDescription AS `Billing Type Description`,
    BillingQuantity AS `Billing Quantity`,
    InvoiceQuantity AS `Invoice Quantity`,
    ActualNetRevenue AS `Actual Net Revenue`,
    DistributionChannel AS `Distribution Channel`,
    TransactionType AS `Transaction Type`,
    ActualGrossProfit AS `Actual Gross Profit`,
    Country AS `Country`,
    Currency AS `Currency`,
    DirectSalesViewFlag AS `Direct Sales View Flag`,
    CustomerViewFlag AS `Customer View Flag`,
    SalesFlag AS `Sales Flag`,
    Source
FROM hub_sbx.s_ipd.s_bowler_billing_master_ww_ipd;